# Template-repeat path matching

This notebook demonstrates the asymmetric `TreePathMatcher` mode

```python
TreePathMatcher(method="exact", mode="template_repeat")
```

The second input `H` is treated as a **path template**. A template vertex may be matched multiple times, but matched template indices must remain nondecreasing along the matched tree path.

The existing behavior remains the default:

```python
TreePathMatcher(method="exact", mode="unique")
```

where a match consumes one vertex from each side.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)

In [ ]:
import numpy as np

from path_matcher.matcher import TreePathMatcher
from path_matcher.tree_data import TreeData


def chain(labels):
    """Create a TreeData path with labels in root-to-leaf order."""
    n = len(labels)
    parent = np.asarray([-1] + list(range(n - 1)), dtype=np.int32)
    return TreeData(
        parent=parent,
        label=list(labels),
        orig_index=np.arange(n, dtype=np.int32),
    )


def strict_label_score(a, b):
    # Use a negative mismatch score in these didactic examples so the displayed
    # paths contain only semantic label matches, not zero-score traceback ties.
    return 1.0 if a == b else -100.0


def describe_alignment(name, G, H, pairs, score):
    print(f"{name}: score={score}")
    print("  pairs:            ", pairs)
    print("  matched G labels: ", [G.label[i] for i, _j in pairs])
    print("  matched H labels: ", [H.label[j] for _i, j in pairs])
    print("  template indices: ", [j for _i, j in pairs])

## 1. Minimal example: unique vs template-repeat

The template is `(1, 2, 3)`. The tree path is `(1, 1, 2, 3, 3, 3)`.

With `mode="unique"`, each template vertex can be used once, so the best score is 3.  
With `mode="template_repeat"`, the first template vertex can match both `1`s and the final template vertex can match all three `3`s, so the best score is 6.

In [ ]:
template = chain([1, 2, 3])
good_tree = chain([1, 1, 2, 3, 3, 3])

unique = TreePathMatcher(
    method="exact",
    mode="unique",  # default; included here for clarity
    w=strict_label_score,
)
repeat = TreePathMatcher(
    method="exact",
    mode="template_repeat",
    w=strict_label_score,
)

pairs_unique, score_unique = unique.predict(good_tree, template)
pairs_repeat, score_repeat = repeat.predict(good_tree, template)

describe_alignment("unique", good_tree, template, pairs_unique, score_unique)
print()
describe_alignment("template_repeat", good_tree, template, pairs_repeat, score_repeat)

## 2. Template order is still enforced

For tree path `(1, 2, 1, 2, 3)`, matching every tree vertex to template `(1, 2, 3)` would require template indices `(0, 1, 0, 1, 2)`, which decreases from `1` back to `0`.

`mode="template_repeat"` therefore returns the best legal subsequence instead of matching all five vertices.

In [ ]:
bad_tree = chain([1, 2, 1, 2, 3])

pairs_bad, score_bad = repeat.predict(bad_tree, template)
describe_alignment("template_repeat on non-monotone example", bad_tree, template, pairs_bad, score_bad)

assert [j for _i, j in pairs_bad] == sorted(j for _i, j in pairs_bad)
assert len(pairs_bad) < bad_tree.n

## 3. Penalizing long repetitions

`template_repeat_penalty` is an optional linear penalty for the second and later use of the same template vertex.

For a run of length `r` against one template vertex, the run contributes

```text
sum(match_scores) - template_repeat_penalty * (r - 1)
```

The first use of a template vertex is not penalized.

In [ ]:
run_tree = chain(["A", "A", "A"])
one_state_template = chain(["A"])

for penalty in [0.0, 0.25, 1.25]:
    matcher = TreePathMatcher(
        method="exact",
        mode="template_repeat",
        w=strict_label_score,
        template_repeat_penalty=penalty,
    )
    pairs, score = matcher.predict(run_tree, one_state_template)
    describe_alignment(f"penalty={penalty}", run_tree, one_state_template, pairs, score)
    print()

## 4. The second input must be a path template

The mode is asymmetric. `G` is the ordinary tree; `H` is the template path. If `H` has branching, the matcher raises a `ValueError`.

In [ ]:
branching_template = TreeData(
    parent=np.asarray([-1, 0, 0], dtype=np.int32),
    label=["root", "left", "right"],
    orig_index=np.arange(3, dtype=np.int32),
)

try:
    repeat.predict(good_tree, branching_template)
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)